In [ ]:
deps_path = '/kaggle/input/datasets/nhhsag12/colpali-dependency'
!pip install --no-index --find-links {deps_path} --requirement {deps_path}/requirements.txt

In [ ]:
import os
import gc
import glob
import json
import pickle
import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.notebook import tqdm

try:
    import pytrec_eval
except Exception:
    pytrec_eval = None

# ==============================================================================
# CONFIG — ColSmol model + ViDoRe data
# ==============================================================================

USE_PREENCODED_INDEX = True
PREENCODED_INDEX_SOURCE = "/kaggle/input/datasets/namthi/colsmol-encoded-vidore/vidore_colsmol_encoded_shards"
FALLBACK_INDEX_PKL_PATH = "/kaggle/input/datasets/namthi/colsmol-encoded-vidore/vidore_colsmol_encoded_shards/vidore_colsmol_manifest.pkl"
INDEX_PKL_PATH = PREENCODED_INDEX_SOURCE if USE_PREENCODED_INDEX else FALLBACK_INDEX_PKL_PATH

VIDORE_DATASET_ROOT = "/kaggle/input/datasets/namthi/vidore-v3"
DOMAIN_FILTER = []  # [] means all domains

# Keep ColSmol model setup unchanged
COLSMOL_BASE = "/kaggle/input/models/nhhsag12/colsmolvlm-instruct-500m-base/pytorch/default/1"
COLSMOL_LORA = "/kaggle/input/models/nhhsag12/colsmol-500m/pytorch/default/4"

WORKING_DIR = "/kaggle/working"
os.makedirs(WORKING_DIR, exist_ok=True)

# Method config (Method 1 & Method 7 common config)
TOPK_RATIOS = [round(i * 0.1, 1) for i in range(1, 10)]

# Cấu hình Method 7 (RVQ) - Chi tiết cấu hình tối ưu RTX PRO 6000 được định nghĩa trực tiếp trong Cell 15
ENABLE_METHOD_7_RVQ = True

# Strict benchmark mode (align with guide behavior)
USE_STRICT_GUIDE_EVAL = True
DEDUP_QUERIES_FOR_EVAL = True
EVAL_BACKEND = "pytrec_eval"   # options: pytrec_eval, fast_formula
AUTO_FALLBACK_WHEN_PYTREC_MISSING = True

print("Config loaded.")
print(f"INDEX_PKL_PATH      : {INDEX_PKL_PATH}")
print(f"VIDORE_DATASET_ROOT : {VIDORE_DATASET_ROOT}")
print(f"COLSMOL_BASE        : {COLSMOL_BASE}")
print(f"COLSMOL_LORA        : {COLSMOL_LORA}")
print(f"TOPK_RATIOS         : {TOPK_RATIOS}")
print(f"ENABLE_METHOD_7_RVQ : {ENABLE_METHOD_7_RVQ}")
print(f"USE_STRICT_GUIDE_EVAL: {USE_STRICT_GUIDE_EVAL}")
print(f"DEDUP_QUERIES_FOR_EVAL: {DEDUP_QUERIES_FOR_EVAL}")
print(f"EVAL_BACKEND         : {EVAL_BACKEND}")
if USE_STRICT_GUIDE_EVAL and EVAL_BACKEND == "pytrec_eval" and pytrec_eval is None:
    if AUTO_FALLBACK_WHEN_PYTREC_MISSING:
        print("[WARN] pytrec_eval not available. Falling back to fast_formula backend.")
        EVAL_BACKEND = "fast_formula"
    else:
        raise ImportError("pytrec_eval is required for strict guide eval. Install it first in this runtime.")
print(f"EVAL_BACKEND(final)  : {EVAL_BACKEND}")

In [ ]:
# Load encoded ViDoRe page embeddings (directory of shards, manifest, or single-file payload)
# ==============================================================================


def _resolve_existing_file(path_like, base_dir=None):
    s = str(path_like)
    candidates = []

    if os.path.isabs(s):
        candidates.append(s)
    elif base_dir:
        candidates.append(os.path.join(base_dir, s))

    if base_dir:
        candidates.append(os.path.join(base_dir, os.path.basename(s)))

    candidates.append(s)

    seen = set()
    for c in candidates:
        c_norm = os.path.normpath(c)
        if c_norm in seen:
            continue
        seen.add(c_norm)
        if os.path.isfile(c_norm):
            return c_norm
    return None


index_source = INDEX_PKL_PATH
raw_embeddings = []
page_keys = []
meta_records = []

if os.path.isdir(index_source):
    shard_files = sorted(glob.glob(os.path.join(index_source, "shard_*.pkl")))
    if not shard_files:
        raise ValueError(f"No shard_*.pkl files found under: {index_source}")

    print(f"Loading sharded index directly from directory: {index_source}")
    print(f"Detected shards: {len(shard_files)}")

    for sp in shard_files:
        with open(sp, "rb") as sf:
            shard = pickle.load(sf)
        raw_embeddings.extend(shard.get("fused_index", []))
        page_keys.extend(shard.get("page_keys", []))
        meta_records.extend(shard.get("metadata", []))

elif os.path.isfile(index_source):
    with open(index_source, "rb") as f:
        payload = pickle.load(f)

    if not isinstance(payload, dict):
        raise ValueError("Expected dict payload in encoded ViDoRe PKL.")

    if payload.get("format") == "vidore_sharded_v1":
        shard_files = payload.get("shard_files", [])
        if not shard_files:
            raise ValueError("Sharded payload has no shard_files.")

        manifest_dir = os.path.dirname(index_source)
        resolved_shards = []
        for sp in shard_files:
            resolved = _resolve_existing_file(sp, base_dir=manifest_dir)
            if resolved is None:
                raise FileNotFoundError(
                    f"Shard not found: {sp} (checked relative to {manifest_dir})"
                )
            resolved_shards.append(resolved)

        print(f"Loading sharded index from manifest: {index_source}")
        print(f"Resolved shards: {len(resolved_shards)}")

        for sp in resolved_shards:
            with open(sp, "rb") as sf:
                shard = pickle.load(sf)
            raw_embeddings.extend(shard.get("fused_index", []))
            page_keys.extend(shard.get("page_keys", []))
            meta_records.extend(shard.get("metadata", []))

    elif any(k in payload for k in ["fused_index", "embeddings"]):
        raw_embeddings = payload.get("fused_index", payload.get("embeddings", []))
        page_keys = payload.get("page_keys", [])
        meta_records = payload.get("metadata", [])

    else:
        raise ValueError("Unsupported index payload format.")

else:
    raise FileNotFoundError(f"Index source not found: {index_source}")

if not raw_embeddings:
    raise ValueError("No embeddings found in index payload.")

meta_df = pd.DataFrame(meta_records) if meta_records else pd.DataFrame()

all_page_embeddings = []
for emb in raw_embeddings:
    if isinstance(emb, torch.Tensor):
        arr = emb.detach().cpu().numpy()
    else:
        arr = np.asarray(emb)
    if arr.ndim != 2:
        raise ValueError(f"Embedding must be 2D (tokens x dim), got {arr.shape}")
    if arr.dtype not in (np.float16, np.float32):
        arr = arr.astype(np.float32, copy=False)
    all_page_embeddings.append(arr)

n_pages = len(all_page_embeddings)
all_page_indices = list(range(n_pages))

if len(meta_df) < n_pages:
    pad_rows = n_pages - len(meta_df)
    meta_df = pd.concat([meta_df, pd.DataFrame(index=range(pad_rows))], ignore_index=True)
meta_df = meta_df.iloc[:n_pages].copy()

if page_keys and len(page_keys) >= n_pages:
    meta_df["page_key"] = page_keys[:n_pages]
elif "page_key" not in meta_df.columns:
    meta_df["page_key"] = [f"page_{i}" for i in range(n_pages)]

if "join_doc_name" not in meta_df.columns:
    meta_df["join_doc_name"] = "unknown_doc"
if "safe_page" not in meta_df.columns:
    meta_df["safe_page"] = np.arange(n_pages, dtype=np.int32)
if "domain" not in meta_df.columns:
    meta_df["domain"] = "unknown"

meta_df["embed_idx"] = np.arange(n_pages, dtype=np.int32)
embedded_rows = meta_df.copy()
avail_docs = set(embedded_rows["join_doc_name"].astype(str).tolist())
doc_page_lookup = {doc: grp for doc, grp in embedded_rows.groupby("join_doc_name")}

print(f"Loaded encoded pages: {n_pages}")
print(f"Embedding shape sample: {all_page_embeddings[0].shape}")
print(f"Embedding dtype sample: {all_page_embeddings[0].dtype}")
print(f"Metadata columns: {list(embedded_rows.columns)}")
print(f"Docs in index: {len(avail_docs)}")

In [ ]:
from colpali_engine.models import ColIdefics3, ColIdefics3Processor
from peft import PeftModel

query_model = ColIdefics3.from_pretrained(
    COLSMOL_BASE,
    torch_dtype=torch.bfloat16,
    device_map="cuda:0",
    attn_implementation="eager" # or eager
)
query_model = PeftModel.from_pretrained(
    query_model,
    COLSMOL_LORA
).eval()
query_processor = ColIdefics3Processor.from_pretrained(COLSMOL_LORA)


In [ ]:
def forward_with_attentions_colpali(model, inputs):
    """
    Run ColPali forward pass and capture per-layer attention weights via hooks.

    Tìm transformer decoder layers bằng cách duyệt toàn bộ submodule thay vì
    hardcode path — tránh AttributeError khi cấu trúc PaliGemma thay đổi theo
    phiên bản transformers.

    Returns:
      proj        : (B, S, dim)  — từ ColPali.forward(), giống Method 1
      raw_outputs : SimpleNamespace với .attentions = tuple of (B, H, S, S) tensors
    """
    from types import SimpleNamespace

    captured_attns = []
    hooks = []

    # Tìm GemmaModel.layers bằng cách duyệt submodules thay vì hardcode path.
    # Với PaliGemmaForConditionalGeneration, cấu trúc thực tế là:
    #   model.model  (PaliGemmaForConditionalGeneration)
    #     .model     (PaliGemmaModel)
    #       .language_model (GemmaForCausalLM)
    #         .model (GemmaModel)
    #           .layers (ModuleList)
    # Nhưng có thể khác nhau tùy phiên bản — nên tìm động.
    layers = None
    candidates = [
        # path đúng nhất theo class ColPali trong notebook này
        lambda m: m.model.model.language_model.model.layers,
        # fallback 1: một số phiên bản bỏ wrapper .model bên trong
        lambda m: m.model.model.language_model.layers,
        # fallback 2: PaliGemmaForConditionalGeneration không có language_model attr
        #             mà dùng .language_model trực tiếp trên .model
        lambda m: m.model.language_model.model.layers,
        lambda m: m.model.language_model.layers,
    ]
    for try_fn in candidates:
        try:
            layers = try_fn(model)
            break
        except AttributeError:
            continue

    # Nếu vẫn không tìm thấy: quét toàn bộ named_modules để tìm ModuleList chứa self_attn
    if layers is None:
        for name, mod in model.named_modules():
            if (isinstance(mod, torch.nn.ModuleList)
                    and len(mod) > 0
                    and hasattr(mod[0], 'self_attn')):
                layers = mod
                break

    if layers is None:
        raise RuntimeError(
            "Không tìm được transformer decoder layers trong model ColPali. "
            "Kiểm tra lại cấu trúc model bằng: print(model)"
        )

    def _attn_hook(module, inp, out):
        # GemmaAttention trả về (attn_output, attn_weights, past_key_value)
        # attn_weights là (B, H, S, S) khi output_attentions=True, ngược lại là None
        if isinstance(out, tuple) and len(out) > 1 and out[1] is not None:
            captured_attns.append(out[1].detach())

    for layer in layers:
        hooks.append(layer.self_attn.register_forward_hook(_attn_hook))

    try:
        with torch.no_grad():
            kwargs = dict(inputs)
            kwargs['output_attentions'] = True
            proj = model(**kwargs)   # ColPali.forward → (B, S, dim)
    finally:
        for h in hooks:
            h.remove()

    raw_outputs = SimpleNamespace(
        attentions=tuple(captured_attns) if captured_attns else None
    )
    return proj, raw_outputs

# ==============================================================================
# Helper: encode_query_live
#
# Encodes a single query string and returns:
#   proj        : (1, S, dim)  L2-normalised, masked  [float32 on device]
#   raw_outputs : ModelOutput  (contains .attentions for Methods 2 & 4)
#   inputs      : BatchEncoding (contains attention_mask, input_ids)
#   encode_ms   : wall-clock encoding time in milliseconds
#
# process_queries() chain (from ColIdefics3Processor source):
#   process_queries([q])
#     → prepends query_prefix ("") + appends pad_token × 10
#     → calls process_texts()
#     → tokenizer(bos_token + text, return_tensors="pt", padding="longest")
# ==============================================================================

def encode_query_live(question: str, processor, model, device: str):
    """
    Tokenise `question` with process_queries() and run a forward pass that
    also captures attention weights.  Used by Methods 2 and 4.
    Returns (proj, raw_outputs, inputs, encode_ms).
    """
    inputs = processor.process_queries([question]).to(device)

    torch.cuda.synchronize()
    t0 = time.perf_counter()

    with torch.no_grad():
        proj, raw_outputs = forward_with_attentions_colpali(model, inputs)

    torch.cuda.synchronize()
    encode_ms = (time.perf_counter() - t0) * 1000.0

    return proj, raw_outputs, inputs, encode_ms

In [ ]:
# DEBUG CELL — run this before QA mapping cell (ViDoRe)
from pathlib import Path

root = Path(VIDORE_DATASET_ROOT)
if not root.exists():
    raise FileNotFoundError(f"ViDoRe dataset root not found: {VIDORE_DATASET_ROOT}")

domain_dirs = [p for p in sorted(root.iterdir()) if p.is_dir()]
if DOMAIN_FILTER:
    domain_dirs = [p for p in domain_dirs if p.name in set(DOMAIN_FILTER)]

print(f"ViDoRe root: {root}")
print(f"Domains selected ({len(domain_dirs)}): {[p.name for p in domain_dirs]}")
print(f"Indexed pages: {len(embedded_rows)}")

if "domain" in embedded_rows.columns:
    print("\nIndexed pages per domain:")
    print(embedded_rows.groupby("domain").size().sort_values(ascending=False).head(20))

query_parquets = []
for d in domain_dirs:
    for fp in d.rglob("*.parquet"):
        pstr = str(fp).replace("\\", "/").lower()
        if "/corpus/" in pstr:
            continue
        query_parquets.append(fp)

print(f"\nNon-corpus parquet files found: {len(query_parquets)}")
for fp in query_parquets[:20]:
    print(" -", fp)

if query_parquets:
    sample_df = pd.read_parquet(query_parquets[0])
    print("\nSample query parquet:", query_parquets[0])
    print("Columns:", list(sample_df.columns))
    print(sample_df.head(2).to_string())

In [ ]:
# ==============================================================================
# Build QA Pairs from ViDoRe (schema-driven, strict)
# ==============================================================================

from pathlib import Path
from collections import defaultdict

print("\nBuilding QA pairs from ViDoRe query files (schema-driven)...")

root = Path(VIDORE_DATASET_ROOT)
if not root.exists():
    raise FileNotFoundError(f"ViDoRe dataset root not found: {VIDORE_DATASET_ROOT}")

domain_dirs = [p for p in sorted(root.iterdir()) if p.is_dir()]
if DOMAIN_FILTER:
    domain_dirs = [p for p in domain_dirs if p.name in set(DOMAIN_FILTER)]

# Build strict corpus_id -> embed_idx mapping from encoded index metadata
if "corpus_id" not in embedded_rows.columns:
    raise ValueError(
        "`corpus_id` column is missing in encoded index metadata. "
        "ViDoRe qrels are keyed by corpus_id, so this mapping is required."
    )

if "domain" not in embedded_rows.columns:
    embedded_rows = embedded_rows.copy()
    embedded_rows["domain"] = "unknown"


def _norm_key(v):
    s = "" if pd.isna(v) else str(v).strip()
    if not s:
        return ""
    try:
        f = float(s)
        if f.is_integer():
            return str(int(f))
    except Exception:
        pass
    return s


def _norm_domain(v):
    return "" if pd.isna(v) else str(v).strip().lower()


corpus_id_to_embed = defaultdict(list)
domain_corpus_id_to_embed = defaultdict(list)

for _, r in embedded_rows.iterrows():
    cid_key = _norm_key(r.get("corpus_id"))
    if not cid_key:
        continue

    d_key = _norm_domain(r.get("domain"))
    eidx = int(r["embed_idx"])

    corpus_id_to_embed[cid_key].append(eidx)
    domain_corpus_id_to_embed[(d_key, cid_key)].append(eidx)

if not corpus_id_to_embed:
    raise ValueError("No valid corpus_id found in encoded metadata.")

for k in list(corpus_id_to_embed.keys()):
    corpus_id_to_embed[k] = sorted(set(corpus_id_to_embed[k]))
for k in list(domain_corpus_id_to_embed.keys()):
    domain_corpus_id_to_embed[k] = sorted(set(domain_corpus_id_to_embed[k]))

qa_pairs = []

scan_files = 0
used_query_files = 0
used_qrels_files = 0

queries_total = 0
qrels_total = 0
qrels_positive = 0
qrels_mapped = 0

missing_qid_in_queries = 0
queries_without_positive_qrels = 0
queries_with_unmapped_corpus = 0

mapped_with_domain_key = 0
mapped_with_global_fallback = 0

for domain_dir in domain_dirs:
    domain_name = domain_dir.name
    domain_key = _norm_domain(domain_name)

    query_frames = []
    qrels_frames = []

    for fp in sorted(domain_dir.rglob("*.parquet")):
        pstr = str(fp).replace("\\", "/").lower()
        if "/corpus/" in pstr:
            continue

        scan_files += 1
        try:
            df = pd.read_parquet(fp)
        except Exception:
            continue

        if df is None or df.empty:
            continue

        if "/queries/" in pstr:
            if "query_id" in df.columns and "query" in df.columns:
                query_frames.append(df)
                used_query_files += 1
            continue

        if "/qrels/" in pstr:
            if "query_id" in df.columns and "corpus_id" in df.columns:
                qrels_frames.append(df)
                used_qrels_files += 1
            continue

    if not query_frames or not qrels_frames:
        continue

    queries_df = pd.concat(query_frames, ignore_index=True)
    qrels_df = pd.concat(qrels_frames, ignore_index=True)

    queries_total += len(queries_df)
    qrels_total += len(qrels_df)

    if "score" in qrels_df.columns:
        qrels_pos = qrels_df[pd.to_numeric(qrels_df["score"], errors="coerce") > 0].copy()
    else:
        qrels_pos = qrels_df.copy()
    qrels_positive += len(qrels_pos)

    # query_id -> list of (corpus_id, relevance_score)
    qid_to_cids = defaultdict(list)
    for _, r in qrels_pos.iterrows():
        qid_key = _norm_key(r.get("query_id"))
        cid_key = _norm_key(r.get("corpus_id"))
        rel_raw = r.get("score", 1)
        try:
            rel_score = float(rel_raw)
        except Exception:
            rel_score = 1.0
        if qid_key and cid_key:
            qid_to_cids[qid_key].append((cid_key, rel_score))

    # Build query_id -> GT embed indices (+ graded relevance per embed_idx)
    qid_to_gt = {}
    qid_to_gt_relevance = {}

    for qid_key, cid_score_pairs in qid_to_cids.items():
        gt = set()
        gt_relevance = {}
        mapped_any = False

        for cid_key, rel_score in cid_score_pairs:
            dom_pair = (domain_key, cid_key)
            if dom_pair in domain_corpus_id_to_embed:
                mapped_any = True
                mapped_indices = domain_corpus_id_to_embed[dom_pair]
                gt.update(mapped_indices)
                for eidx in mapped_indices:
                    gt_relevance[eidx] = max(float(gt_relevance.get(eidx, 0.0)), float(rel_score))
                mapped_with_domain_key += 1
                continue

            if cid_key in corpus_id_to_embed:
                mapped_any = True
                mapped_indices = corpus_id_to_embed[cid_key]
                gt.update(mapped_indices)
                for eidx in mapped_indices:
                    gt_relevance[eidx] = max(float(gt_relevance.get(eidx, 0.0)), float(rel_score))
                mapped_with_global_fallback += 1

        if mapped_any and gt:
            qid_to_gt[qid_key] = sorted(int(x) for x in gt)
            qid_to_gt_relevance[qid_key] = {int(k): float(v) for k, v in gt_relevance.items() if float(v) > 0}
            qrels_mapped += 1

    for _, r in queries_df.iterrows():
        qid_key = _norm_key(r.get("query_id"))
        if not qid_key:
            missing_qid_in_queries += 1
            continue

        qtext = r.get("query")
        question = "" if pd.isna(qtext) else str(qtext).strip()
        if not question:
            continue

        if qid_key not in qid_to_cids:
            queries_without_positive_qrels += 1
            continue

        gt_indices = qid_to_gt.get(qid_key, [])
        gt_relevance = qid_to_gt_relevance.get(qid_key, {})
        if not gt_indices:
            queries_with_unmapped_corpus += 1
            continue

        if "join_doc_name" in embedded_rows.columns and len(gt_indices) > 0:
            doc_name = str(embedded_rows.iloc[int(gt_indices[0])].get("join_doc_name", domain_name))
        else:
            doc_name = domain_name

        qa_pairs.append(
            {
                "question": question,
                "gt_embed_indices": gt_indices,
                "gt_relevance": gt_relevance,
                "doc_name": doc_name,
                "domain": domain_name,
            }
        )

qa_pairs_before_dedup = len(qa_pairs)
if DEDUP_QUERIES_FOR_EVAL:
    deduped = []
    seen_questions = set()
    for row in qa_pairs:
        q_key = str(row.get("question", "")).strip()
        if not q_key:
            continue
        if q_key in seen_questions:
            continue
        seen_questions.add(q_key)
        deduped.append(row)
    qa_pairs = deduped

print(f"Parquet files scanned            : {scan_files}")
print(f"Query files used                : {used_query_files}")
print(f"Qrels files used                : {used_qrels_files}")
print(f"Queries rows total              : {queries_total}")
print(f"Qrels rows total                : {qrels_total}")
print(f"Qrels rows positive             : {qrels_positive}")
print(f"Qrels query_ids mapped to index : {qrels_mapped}")
print(f"QA pairs built                  : {len(qa_pairs)}")
print(f"QA pairs deduplicated removed   : {qa_pairs_before_dedup - len(qa_pairs)}")
print(f"Queries w/o positive qrels      : {queries_without_positive_qrels}")
print(f"Queries unmapped corpus_id      : {queries_with_unmapped_corpus}")
print(f"Queries missing query_id        : {missing_qid_in_queries}")

print(f"Mapped by (domain, corpus_id)   : {mapped_with_domain_key}")
print(f"Mapped by global fallback       : {mapped_with_global_fallback}")

if not qa_pairs:
    raise ValueError(
        "No QA pairs built from strict query_id/corpus_id mapping. "
        "This indicates index metadata corpus_id is not aligned with qrels corpus_id."
    )

In [ ]:
# ==============================================================================
# Shared utilities: doc matrix builder, MaxSim, metrics, latency tracker
# ==============================================================================

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")


def build_doc_matrix(embeddings, device):
    """
    Convert list of np.ndarray embeddings to a padded (n_docs, max_len, D) tensor.
    Returns (doc_matrix, doc_mask).
    """
    arrays   = [torch.from_numpy(e).float() for e in embeddings]
    max_len  = max(a.shape[0] for a in arrays)
    D        = arrays[0].shape[1]
    n        = len(arrays)
    mat      = torch.zeros(n, max_len, D, dtype=torch.float32)
    mask     = torch.zeros(n, max_len, dtype=torch.bool)
    for i, a in enumerate(arrays):
        L = a.shape[0]
        mat[i, :L] = F.normalize(a, dim=-1)
        mask[i, :L] = True
    return mat.to(device), mask.to(device)


@torch.no_grad()
def fast_maxsim(q_norm, doc_matrix, doc_mask):
    """
    q_norm   : (N_q, D)   — query tokens, L2-normalized
    doc_matrix: (n_docs, max_len, D)
    doc_mask : (n_docs, max_len)  bool
    Returns  : (N_q, n_docs)  per-token MaxSim scores
    """
    sim = torch.einsum('qd,nld->qnl', q_norm, doc_matrix)   # (N_q, n_docs, max_len)
    sim.masked_fill_(~doc_mask.unsqueeze(0), float('-inf'))
    return sim.max(dim=-1).values                             # (N_q, n_docs)


# ---------- Metrics ----------

def _normalize_gt_relevance(gt):
    # Accept dict[int,float] (graded qrels) or list/set of indices (binary relevance).
    if isinstance(gt, dict):
        out = {}
        for k, v in gt.items():
            try:
                kk = int(k)
                vv = float(v)
            except Exception:
                continue
            if vv > 0:
                out[kk] = vv
        return out

    out = {}
    for x in gt:
        try:
            xx = int(x)
            out[xx] = 1.0
        except Exception:
            continue
    return out

def recall_at_k(ranked, gt, k):
    gt_rel = _normalize_gt_relevance(gt)
    if not gt_rel:
        return 0.0
    denom = float(len(gt_rel))
    num = sum(1 for i in ranked[:k] if int(i) in gt_rel)
    return float(num) / denom if denom > 0 else 0.0

def compute_ndcg(ranked, gt, k):
    gt_rel = _normalize_gt_relevance(gt)
    if not gt_rel:
        return 0.0

    dcg = 0.0
    for r, i in enumerate(ranked[:k]):
        gain = float(gt_rel.get(int(i), 0.0))
        if gain > 0:
            dcg += gain / np.log2(r + 2)

    ideal_gains = sorted(gt_rel.values(), reverse=True)[:k]
    idcg = sum(float(g) / np.log2(r + 2) for r, g in enumerate(ideal_gains))
    return dcg / idcg if idcg > 0 else 0.0

def hit_metrics(top10, gt):
    gt_rel = _normalize_gt_relevance(gt)

    if USE_STRICT_GUIDE_EVAL and EVAL_BACKEND == "pytrec_eval":
        if pytrec_eval is None:
            raise ImportError("pytrec_eval backend requested but package is unavailable.")

        qrels = {"q0": {f"d{int(doc_id)}": float(rel) for doc_id, rel in gt_rel.items() if float(rel) > 0}}

        # Synthetic monotonic scores preserve rank order for pytrec_eval.
        results = {"q0": {f"d{int(doc_id)}": float(len(top10) - rank) for rank, doc_id in enumerate(top10)}}

        evaluator = pytrec_eval.RelevanceEvaluator(qrels, {"recall.1,5,10", "ndcg_cut.1,5,10"})
        s = evaluator.evaluate(results)["q0"]
        return {
            'r1':  float(s.get('recall_1', 0.0)),
            'r5':  float(s.get('recall_5', 0.0)),
            'r10': float(s.get('recall_10', 0.0)),
            'n1':  float(s.get('ndcg_cut_1', 0.0)),
            'n5':  float(s.get('ndcg_cut_5', 0.0)),
            'n10': float(s.get('ndcg_cut_10', 0.0)),
        }

    return {
        'r1':  float(recall_at_k(top10, gt_rel, 1)),
        'r5':  float(recall_at_k(top10, gt_rel, 5)),
        'r10': float(recall_at_k(top10, gt_rel, 10)),
        'n1':  float(compute_ndcg(top10, gt_rel, 1)),
        'n5':  float(compute_ndcg(top10, gt_rel, 5)),
        'n10': float(compute_ndcg(top10, gt_rel, 10)),
    }

def _init_metric():
    return {'r1': 0, 'r5': 0, 'r10': 0, 'n1': 0.0, 'n5': 0.0, 'n10': 0.0, 'count': 0}

def _add_metric(dst, src):
    dst['r1']    += float(src['r1'])
    dst['r5']    += float(src['r5'])
    dst['r10']   += float(src['r10'])
    dst['n1']    += float(src['n1'])
    dst['n5']    += float(src['n5'])
    dst['n10']   += float(src['n10'])
    dst['count'] += 1

def _ensure(store, key):
    if key not in store: store[key] = _init_metric()
    return store[key]

def record(all_metrics, all_domain_metrics, key, m, domain):
    _add_metric(_ensure(all_metrics, key), m)
    if domain not in all_domain_metrics:
        all_domain_metrics[domain] = {}
    _add_metric(_ensure(all_domain_metrics[domain], key), m)


def print_summary(all_metrics, all_domain_metrics, method_keys, title=""):
    if title:
        print(f"\n{'='*60}\n{title}\n{'='*60}")
    print(f"{'Method':<35} {'R@1':>7} {'R@5':>7} {'R@10':>7} {'nDCG@1':>8} {'nDCG@5':>8} {'nDCG@10':>9}")
    print("-" * 92)
    for key in method_keys:
        if key not in all_metrics: continue
        m = all_metrics[key]; cnt = m['count'] or 1
        print(f"{key:<35} {m['r1']/cnt*100:6.2f}%  {m['r5']/cnt*100:6.2f}%  "
              f"{m['r10']/cnt*100:6.2f}%  {m['n1']/cnt:7.4f}  {m['n5']/cnt:7.4f}  {m['n10']/cnt:8.4f}")


# ---------- Latency Tracker ----------

class LatencyTracker:
    """
    Tracks per-ratio latency of MaxSim scoring AFTER pooling/pruning.

    Only measures the time for: MaxSim scoring + aggregation + top-k
    on the already-reduced multi-vector representation.
    Excludes model forward pass, pooling, and pruning computation.

    Usage:
        tracker = LatencyTracker("Hierarchical Ward Pool")
        tracker.add_ratio(ratio, score_ms)   # inside loop, per ratio
        tracker.report()                     # after loop
    """
    def __init__(self, method_name: str):
        self.name = method_name
        self.ratio_ms = {}   # ratio -> list[float]  pool+retrieve ms per query

    def add_ratio(self, ratio: float, score_ms: float):
        """Record scoring latency for a single (ratio, query) observation."""
        if ratio not in self.ratio_ms:
            self.ratio_ms[ratio] = []
        self.ratio_ms[ratio].append(score_ms)

    def report(self):
        if not self.ratio_ms:
            print(f"[{self.name}] No latency data collected.")
            return
        print(f"\n{'='*70}")
        print(f"Latency Report — {self.name}  (scoring only, post-pooling/pruning)")
        print(f"{'='*70}")
        print(f"  {'Ratio':<10} {'n':>6} {'avg ms':>10} {'p50 ms':>10} {'p95 ms':>10}")
        print(f"  {'-'*50}")
        for ratio in sorted(self.ratio_ms.keys()):
            vals = self.ratio_ms[ratio]
            n    = len(vals)
            avg  = np.mean(vals)
            p50  = np.percentile(vals, 50)
            p95  = np.percentile(vals, 95)
            print(f"  {ratio:<10.0%} {n:>6} {avg:>10.2f} {p50:>10.2f} {p95:>10.2f}")

    def to_dict(self):
        rows = []
        for ratio in sorted(self.ratio_ms.keys()):
            vals = self.ratio_ms[ratio]
            n    = len(vals)
            rows.append({
                'method':           self.name,
                'ratio':            ratio,
                'n_queries':        n,
                'avg_score_ms':     round(np.mean(vals), 3) if n else 0,
                'p50_score_ms':     round(np.percentile(vals, 50), 3) if n else 0,
                'p95_score_ms':     round(np.percentile(vals, 95), 3) if n else 0,
            })
        return rows


# ==============================================================================
# Spherical KMeans (cosine similarity) — Method 5
# Input : X (N, D) L2-normalized query token vectors
# Output: centroids (K, D) — mean-pool each cluster then re-normalize
# ==============================================================================

def spherical_kmeans(X, K, n_iters=10):
    """
    Cluster N query token vectors into K representative centroids
    using cosine distance (spherical KMeans).

    Args:
        X      : (N, D)  L2-normalized query token vectors
        K      : int     number of clusters (representative tokens to keep)
        n_iters: int     number of Lloyd-style iterations

    Returns:
        centroids: (K, D)  L2-normalized cluster centroids
    """
    N, D = X.shape
    K = min(K, N)

    if K >= N:
        return X.clone()   # every token is its own representative

    # Initialise: pick K distinct tokens at random
    perm      = torch.randperm(N, device=X.device)
    centroids = X[perm[:K]].clone()   # (K, D)

    for _ in range(n_iters):
        # Assignment: cosine sim = dot product when X is normalised
        sim         = torch.mm(X, centroids.t())   # (N, K)
        cluster_ids = sim.argmax(dim=1)             # (N,)

        # Update: mean-pool members → re-normalize
        new_centroids = torch.zeros_like(centroids)
        for c in range(K):
            members = (cluster_ids == c).nonzero(as_tuple=True)[0]
            new_centroids[c] = (X[members].mean(dim=0)
                                if members.numel() > 0 else centroids[c])
        centroids = F.normalize(new_centroids, dim=-1)

    return centroids   # (K, D)


# ==============================================================================
# Random Token Pruning — Method 6
# Randomly sample round(M * ratio) content tokens; average over N_RANDOM_SEEDS.
# ==============================================================================

def random_prune_topk(q_norm, content_idx, doc_matrix, doc_mask,
                      topk_ratios, n_seeds=1):
    """
    For each ratio r keep k = round(M * r) randomly selected tokens.
    Scores are averaged over `n_seeds` independent random draws to reduce
    variance.

    Timing covers the full scoring work per ratio:
      random sampling + MaxSim + sum  (averaged over n_seeds).
    This matches how all other methods measure latency.

    Args:
        q_norm      : (M, D)   content token vectors, L2-normalized
        content_idx : (M,)     positions of content tokens in the full sequence
                               (unused here; kept for API symmetry)
        doc_matrix  : (n_docs, max_len, D)
        doc_mask    : (n_docs, max_len)  bool
        topk_ratios : list[float]  fractions of tokens to KEEP
        n_seeds     : int  number of random samples to average

    Returns:
        results : dict {ratio: scores (n_docs,)}
        timing  : dict {ratio: score_ms}   ms for sampling + MaxSim per ratio
    """
    M      = q_norm.shape[0]
    n_docs = doc_matrix.shape[0]
    device = q_norm.device

    if M == 0:
        zero = torch.zeros(n_docs, device=device)
        return {r: zero for r in topk_ratios}, {r: 0.0 for r in topk_ratios}

    results = {}
    timing  = {}

    for ratio in topk_ratios:
        k = max(1, round(M * ratio))
        k = min(k, M)

        # ── Time: random sample + MaxSim + sum (all scoring work) ────────
        torch.cuda.synchronize()
        t0 = time.perf_counter()

        if k == M:
            # Keep all tokens — no pruning needed; score as-is
            scores = fast_maxsim(q_norm, doc_matrix, doc_mask).sum(dim=0)
        else:
            accumulated = torch.zeros(n_docs, device=device, dtype=torch.float32)
            for _ in range(n_seeds):
                perm        = torch.randperm(M, device=device)[:k]
                pruned      = q_norm[perm]          # (k, D)
                accumulated += fast_maxsim(pruned, doc_matrix, doc_mask).sum(dim=0)
            scores = accumulated / n_seeds

        torch.cuda.synchronize()
        timing[ratio] = (time.perf_counter() - t0) * 1000.0
        # ─────────────────────────────────────────────────────────────────

        results[ratio] = scores

    return results, timing


print("Utility functions ready.")

In [ ]:
# ==============================================================================
# METHOD 1 — Traditional MaxSim
# Queries encoded live with ColPali (plain forward, no attention capture).
# ==============================================================================

print(">>> METHOD 1: Traditional MaxSim")

trad_metrics        = {}
trad_domain_metrics = {}
trad_query_rows     = []
trad_latency        = LatencyTracker("Traditional MaxSim")

METHOD_KEYS_TRAD = ['traditional']

# Build full doc matrix from all page embeddings
print(f"Building doc matrix from {len(all_page_embeddings)} page embeddings...")
doc_matrix, doc_mask = build_doc_matrix(all_page_embeddings, device)
n_docs = doc_matrix.shape[0]
print(f"Doc matrix shape: {doc_matrix.shape}")

pbar = tqdm(enumerate(qa_pairs), total=len(qa_pairs), desc="Traditional MaxSim")

for q_idx, item in pbar:
    question = item['question']
    gt_set   = item.get('gt_relevance', item['gt_embed_indices'])
    domain   = item['domain']

    # ── Encode query live ──────────────────────────────────────────────
    q_inputs = query_processor.process_queries([question]).to(device)

    with torch.no_grad():
        q_proj = query_model(**q_inputs)   # (1, S, dim)  — plain forward

    attn_mask = q_inputs['attention_mask'][0]                       # (S,)
    trad_idx  = torch.where(attn_mask > 0)[0]
    q_emb     = q_proj[0][trad_idx].float()                         # (N, dim)
    q_norm    = F.normalize(q_emb, dim=-1)

    # ── Retrieval (timed — this is the 100% baseline) ───────────────────
    torch.cuda.synchronize()
    t_score_start = time.perf_counter()

    M      = fast_maxsim(q_norm, doc_matrix, doc_mask)               # (N, n_docs)
    scores = M.sum(dim=0)
    top10  = torch.topk(scores, min(10, n_docs)).indices.cpu().tolist()

    torch.cuda.synchronize()
    score_ms = (time.perf_counter() - t_score_start) * 1000.0
    trad_latency.add_ratio(1.0, score_ms)

    m = hit_metrics(top10, gt_set)
    record(trad_metrics, trad_domain_metrics, 'traditional', m, domain)

    trad_query_rows.append({
        'query_id':       q_idx,
        'doc_name':       item['doc_name'],
        'domain':         domain,
        'question':       question,
        'trad_r@1':       m['r1'],
        'trad_r@5':       m['r5'],
        'trad_r@10':      m['r10'],
        'trad_ndcg@1':    round(m['n1'],  4),
        'trad_ndcg@5':    round(m['n5'],  4),
        'trad_ndcg@10':   round(m['n10'], 4),
    })

print_summary(trad_metrics, trad_domain_metrics, METHOD_KEYS_TRAD,
              title="Traditional MaxSim Results")
trad_latency.report()

pd.DataFrame(trad_query_rows).to_csv(
    os.path.join(WORKING_DIR, "traditional_queries.csv"), index=False)
print("\n✅ Saved: traditional_queries.csv")

In [ ]:
# ==============================================================================
# METHOD 7 - RVQ Training on ColPali Train Set
# Train RVQ codebooks from scratch using the official ColPali training dataset.
#
# Variables inherited from earlier cells (no need to redefine):
#   Cell 1: os, gc, glob, json, time, np, torch, F, tqdm
#            COLSMOL_BASE, COLSMOL_LORA, WORKING_DIR, device
#   Cell 4: query_model (ColSmol + LoRA, bfloat16, on GPU)
#            query_processor (ColIdefics3Processor)
# ==============================================================================

# -- Install vector-quantize-pytorch (offline wheel preferred) -----------------
import subprocess, io
_rvq_wheel_dir = "/kaggle/input/datasets/thinam4/rvq-wheels/rvq_wheels"
if os.path.isdir(_rvq_wheel_dir):
    subprocess.run(
        ["pip", "install", "--quiet", "--no-index",
         "--find-links", _rvq_wheel_dir, "vector-quantize-pytorch"],
        check=True
    )
else:
    subprocess.run(["pip", "install", "--quiet", "vector-quantize-pytorch"], check=True)

from vector_quantize_pytorch import ResidualVQ

# ==============================================================================
# CONFIG - RTX PRO 6000 (Blackwell, 96 GB VRAM)
# ==============================================================================

# Training data: ColPali official train split (82 shards)
TRAIN_DATA_ROOT = "/kaggle/input/datasets/namthi/colpali-train-set"

# Output codebooks go into /kaggle/working/rvq_codebooks
# WORKING_DIR is already defined in Cell 1 as "/kaggle/working"
OUTPUT_DIR = os.path.join(WORKING_DIR, "rvq_codebooks")
os.makedirs(OUTPUT_DIR, exist_ok=True)

EMB_DIM = 128   # ColPali projection dimension (query_model.dim == 128)

# RVQ configs to train: (NQ, CB_SIZE, label)
#   c32_f32 -> 32 quantizers x 32-code codebook -> 32 bytes/patch  (4x compression)
RVQ_CONFIGS = [
    (32, 32, "c32_f32"),
]


RVQ_TRAINING_EPOCHS     = 40       # max epochs per config (early-stop patience=10)
RVQ_TRAINING_BATCH_SIZE = 8192     # gradient-step batch size (tuned for 96 GB VRAM)
N_TRAIN_PATCHES         = 250_000  # total patches to collect across all shards
PATCHES_PER_SHARD       = 3_000    # per-shard cap - prevents any domain dominating
ENCODE_BATCH_SIZE       = 16       # images per ColPali forward pass

# Blackwell / RTX PRO 6000: enable TF32 for ~2x GEMM throughput
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

ENCODE_DTYPE = torch.bfloat16  # must match query_model dtype from Cell 4

print(">>>> METHOD 7: RVQ Codebook Training")
print(f"  Model base     : {COLSMOL_BASE}")
print(f"  Model LoRA     : {COLSMOL_LORA}")
print(f"  Train data     : {TRAIN_DATA_ROOT}")
print(f"  Output         : {OUTPUT_DIR}")
print(f"  Configs        : {[c[2] for c in RVQ_CONFIGS]}")
print(f"  Target patches : {N_TRAIN_PATCHES:,}")
print(f"  Batch size     : {RVQ_TRAINING_BATCH_SIZE:,}")
print()

# ==============================================================================
# DATA STREAMING - lazy Parquet shard loader
# ==============================================================================

def _list_train_shards(root):
    # Return sorted list of train-*.parquet shard paths
    pattern = os.path.join(root, "train-*.parquet")
    shards  = sorted(glob.glob(pattern))
    if not shards:
        raise FileNotFoundError(f"No train shards found at {pattern}")
    return shards


def _load_shard_images(shard_path):
    # Load one Parquet shard lazily and return a list of PIL Images.
    # Handles 'image' column as: bytes, dict{'bytes':...}, or PIL Image directly.
    import pyarrow.parquet as pq
    from PIL import Image

    table  = pq.read_table(shard_path, columns=["image"])
    images = []
    for row in table.to_pydict()["image"]:
        if isinstance(row, dict):
            img_bytes = row.get("bytes") or row.get("path")
            if isinstance(img_bytes, str):
                img = Image.open(img_bytes).convert("RGB")
            else:
                img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
        elif isinstance(row, (bytes, bytearray)):
            img = Image.open(io.BytesIO(row)).convert("RGB")
        else:
            img = row.convert("RGB")
        images.append(img)
    return images


@torch.no_grad()
def _encode_images_rvq(images, batch_size=ENCODE_BATCH_SIZE):
    all_embs = []
    for start in range(0, len(images), batch_size):
        batch_imgs = images[start : start + batch_size]
        inputs     = query_processor.process_images(batch_imgs)
        inputs     = {k: v.to(device) for k, v in inputs.items()}
        with torch.autocast(device_type="cuda", dtype=ENCODE_DTYPE):
            embs = query_model(**inputs)
        embs = embs.float().cpu().numpy()
        del inputs          # ← xóa inputs TRƯỚC khi append để giải phóng VRAM ngay
        torch.cuda.empty_cache()
        gc.collect()
        for i in range(embs.shape[0]):
            all_embs.append(embs[i])
        del embs
    return all_embs


# ==============================================================================
# DOMAIN-AWARE PATCH SAMPLER
# ==============================================================================

def top_norm_filter(patches: np.ndarray, keep_ratio: float = 0.5) -> np.ndarray:
    """Keep only the top-`keep_ratio` patches by L2 norm.
    Low-norm patches are typically blank/whitespace regions and hurt codebook quality.
    Operates in raw (un-normalised) Euclidean space — do NOT call after normalisation.
    """
    norms = np.linalg.norm(patches, axis=1)          # (N,)
    k     = max(1, int(len(norms) * keep_ratio))
    idx   = np.argpartition(norms, -k)[-k:]          # top-k indices (unordered)
    return patches[idx]


def collect_training_patches(shards, target=N_TRAIN_PATCHES,
                              patches_per_shard=PATCHES_PER_SHARD, seed=42):
    # Stream shards -> encode -> filter whitespace patches (top 50% by L2 norm)
    # -> random subsample -> globally shuffled patch array.
    # Per-shard cap avoids over-representation of any single document domain.
    # NOTE: patches are kept in raw Euclidean space (NOT L2-normalised) so that
    #       ResidualVQ can learn geometry-correct Euclidean codebooks.
    # Returns: np.ndarray (N, EMB_DIM) float32
    rng          = np.random.default_rng(seed)
    patch_buf    = []
    total_so_far = 0

    print(f"Collecting patches: {len(shards)} shards, "
          f"cap {patches_per_shard:,}/shard, target {target:,} ...\n")

    pbar = tqdm(shards, desc="Encoding shards", unit="shard")
    for shard_path in pbar:
        if total_so_far >= target:
            break
        try:
            images = _load_shard_images(shard_path)
        except Exception as e:
            print(f"  [WARN] Skip {os.path.basename(shard_path)}: {e}")
            continue

        embs = _encode_images_rvq(images)
        del images; gc.collect()

        shard_patches = np.concatenate(embs, axis=0).astype(np.float32)
        del embs

        # ── Step 1: filter out low-norm (whitespace/blank) patches ──────────
        shard_patches = top_norm_filter(shard_patches, keep_ratio=0.5)

        # ── Step 2: subsample to per-shard cap ──────────────────────────────
        n_shard = shard_patches.shape[0]
        n_take  = min(patches_per_shard, n_shard, target - total_so_far)
        if n_take < n_shard:
            idx           = rng.choice(n_shard, n_take, replace=False)
            shard_patches = shard_patches[idx]

        # ── NOTE: No L2-normalisation here. Euclidean space is preserved. ───

        patch_buf.append(shard_patches)
        total_so_far += shard_patches.shape[0]
        pbar.set_postfix({"patches": f"{total_so_far:,}"})

    all_patches = np.concatenate(patch_buf, axis=0)
    shuffle_idx = rng.permutation(all_patches.shape[0])
    print(f"\n  Total patches: {all_patches.shape[0]:,}  ({all_patches.nbytes/1e6:.1f} MB)")
    return all_patches[shuffle_idx]


# ==============================================================================
# RVQ TRAINING HELPERS
# ==============================================================================

def train_rvq_model(NQ, CB_SIZE, train_data):
    # Train ResidualVQ model.
    # train_data: (N, D) float32 CPU tensor, L2-normalised.
    # Returns trained model moved back to CPU.
    rvq_model = ResidualVQ(
        dim                     = EMB_DIM,
        num_quantizers          = NQ,
        codebook_size           = CB_SIZE,
        kmeans_init             = True,       # warm-start centroids with k-means
        threshold_ema_dead_code = 2,          # revive dead codebook entries automatically
        commitment_weight       = 0.25,
    ).to(device)
    rvq_model.train()

    n = train_data.shape[0]
    print(f"  Training NQ={NQ}, CB_SIZE={CB_SIZE} | {n:,} patches | max {RVQ_TRAINING_EPOCHS} epochs")

    best_loss, no_improve = float("inf"), 0

    for epoch in range(RVQ_TRAINING_EPOCHS):
        perm      = torch.randperm(n)
        ep_loss   = 0.0
        n_batches = 0

        for i in range(0, n, RVQ_TRAINING_BATCH_SIZE):
            batch = train_data[perm[i : i + RVQ_TRAINING_BATCH_SIZE]].to(device)
            with torch.no_grad():     # không cần grad, EMA update xảy ra trong forward
                _, _, commit_loss = rvq_model(batch.unsqueeze(1))
            loss_val = commit_loss.sum().item()
            ep_loss   += loss_val
            n_batches += 1

        avg_loss = ep_loss / max(n_batches, 1)

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"    Epoch {epoch+1:>3}/{RVQ_TRAINING_EPOCHS}: commit_loss={avg_loss:.6f}")

        # Early stopping: patience=10
        if avg_loss < best_loss - 1e-6:
            best_loss, no_improve = avg_loss, 0
        else:
            no_improve += 1
        if no_improve >= 10:
            print(f"    Early stop at epoch {epoch+1} (no improvement for 10 epochs)")
            break

    rvq_model.eval()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    return rvq_model


def extract_rvq_codebooks(rvq_model, NQ, CB_SIZE):
    # Extract codebook tensors from a trained ResidualVQ.
    # Codebooks are kept in raw Euclidean space (NO L2-normalisation) so that
    # inference-time distance computations (Euclidean / ADC) remain correct.
    # Returns np.ndarray (NQ, CB_SIZE, EMB_DIM) float32.
    cbs = []
    for qi in range(NQ):
        vq = rvq_model.layers[qi]
        cb = None
        for attr_path in ['_codebook.embed', 'codebook', '_codebook.embed_avg']:
            try:
                obj = vq
                for part in attr_path.split('.'):
                    obj = getattr(obj, part)
                if isinstance(obj, torch.Tensor):
                    cb = obj[0] if obj.ndim == 3 else obj
                    cb = cb.detach().float().cpu()   # NO F.normalize — preserve Euclidean geometry
                    break
            except (AttributeError, IndexError):
                continue
        if cb is None:
            raise RuntimeError(
                f"Cannot extract codebook for quantizer {qi}. "
                "Try updating vector-quantize-pytorch."
            )
        cbs.append(cb.numpy())
    return np.stack(cbs, axis=0)   # (NQ, CB_SIZE, EMB_DIM)


def save_rvq_codebook(cbs_np, NQ, CB_SIZE, label):
    # Save codebook as .npy + metadata .json to OUTPUT_DIR.
    npy_path  = os.path.join(OUTPUT_DIR, f"{label}_codebook.npy")
    meta_path = os.path.join(OUTPUT_DIR, f"{label}_meta.json")

    np.save(npy_path, cbs_np)

    meta = {
        "label"               : label,
        "NQ"                  : NQ,
        "CB_SIZE"             : CB_SIZE,
        "EMB_DIM"             : EMB_DIM,
        "codebook_shape"      : list(cbs_np.shape),
        "bytes_per_patch"     : NQ,
        "compression_vs_f32"  : round(EMB_DIM * 4 / NQ, 1),
        "trained_on"          : f"{N_TRAIN_PATCHES:,} patches - ColPali train set",
        "model_base"          : COLSMOL_BASE,
        "model_lora"          : COLSMOL_LORA,
        "timestamp"           : time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }
    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=2)

    size_mb = os.path.getsize(npy_path) / 1e6
    print(f"  OK: {npy_path}  ({size_mb:.3f} MB)")
    print(f"     meta : {meta_path}")
    print(f"     shape: {cbs_np.shape}  dtype={cbs_np.dtype}")


# ==============================================================================
# MAIN - collect patches -> free encoder -> train each config -> save
# ==============================================================================

_t_total = time.time()

# Step 1: Discover training shards
_shards = _list_train_shards(TRAIN_DATA_ROOT)
print(f"Found {len(_shards)} train shards\n")

# Step 2: Encode images + domain-aware sampling using query_model (Cell 4)
_patches_np = collect_training_patches(_shards)
_train_data = torch.from_numpy(_patches_np)
del _patches_np; gc.collect()

# Step 3: Move query_model to CPU to free ~8 GB VRAM for RVQ training.
#   NOTE: query_model is NOT deleted. Restore with query_model.cuda() if needed later.
with torch.no_grad():
    pass  # flush autograd graph
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    free_gb = (torch.cuda.get_device_properties(0).total_memory
               - torch.cuda.memory_allocated()) / 1e9
    print(f"VRAM free: {free_gb:.1f} GB — proceeding to RVQ training.\n")
gc.collect()

# Step 4: Train each RVQ config and save codebook to disk
_rvq_results = {}
for _NQ, _CB_SIZE, _label in RVQ_CONFIGS:
    print(f"\n{'='*60}")
    print(f"CONFIG: {_label}  (NQ={_NQ}, CB_SIZE={_CB_SIZE})")
    print(f"{'='*60}")

    _t_cfg   = time.time()
    _rvq     = train_rvq_model(_NQ, _CB_SIZE, _train_data)
    _elapsed = time.time() - _t_cfg
    print(f"  Training time: {_elapsed:.1f}s")

    _cbs_np = extract_rvq_codebooks(_rvq, _NQ, _CB_SIZE)
    del _rvq; gc.collect()

    save_rvq_codebook(_cbs_np, _NQ, _CB_SIZE, _label)
    del _cbs_np; gc.collect()

    _rvq_results[_label] = {"NQ": _NQ, "CB_SIZE": _CB_SIZE, "train_secs": round(_elapsed, 1)}

del _train_data; gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Step 5: Print summary table
print(f"\n{'='*60}")
print(f"ALL DONE in {(time.time()-_t_total)/60:.1f} min")
print(f"{'='*60}")
print(f"{'Config':<15} {'NQ':>4} {'CB':>5} {'bytes/patch':>12} {'ratio':>8} {'secs':>8}")
print("-" * 55)
for _lbl, _r in _rvq_results.items():
    _ratio = round(EMB_DIM * 4 / _r["NQ"], 1)
    print(f"{_lbl:<15} {_r['NQ']:>4} {_r['CB_SIZE']:>5} {_r['NQ']:>12}  {_ratio:>6.1f}x  {_r['train_secs']:>7.1f}")

print(f"\nCodebooks saved to: {OUTPUT_DIR}")
print("  -> Download .npy files from Kaggle Output tab,")
print("     then upload as a dataset to reuse in inference notebooks.")
print("\nFiles:")
for _fn in sorted(os.listdir(OUTPUT_DIR)):
    _fp = os.path.join(OUTPUT_DIR, _fn)
    print(f"  {_fn}  ({os.path.getsize(_fp)/1e6:.3f} MB)")